# Week 2 Live Coding
## From a gap to a coefficient

Three parts:

1. **Voter-file slice.** A raw gap of **37 percentage points** between mailed and non-mailed voters that almost entirely disappears once we look at voters who are actually comparable.
2. **District dataset.** The vendor's chart. We reproduce the 8.7-point gap, then subtract out the previous cycle's turnout.
3. **The same numbers, as a regression.** A scatterplot and correlation, then the 8.7 as a regression coefficient.

New tools this week:
- `df.shape`: how many rows and columns does this table have?
- `df.groupby(col)[outcome].mean()`: compute an average of `outcome` separately for each value of `col`.
- `df.groupby([col1, col2])[outcome].mean()`: same thing, but split by two columns at once.
- `df['new'] = df['a'] - df['b']`: create a new column from existing ones.
- `plt.scatter(...)` and `df['a'].corr(df['b'])`: see and measure a relationship between two columns.
- `smf.ols('y ~ x', data=df).fit()`: fit a regression; read its coefficients. (All pre-filled. You run it.)

**One thing to know before you start.** Turnout in these files is stored as a **proportion**, not a percentage. `0.630` means 63.0%. Multiply by 100 to read any of these numbers in percentage points.

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

Run the cell below to load the data.

In [ ]:
import pandas as pd

## Part 1: The voter-file slice

2,000 voters. For each voter we know three things:

- `past_vote_score`: how many of the last 4 elections this person voted in (0 through 4). A rough measure of how likely this voter is to turn out, based only on their history.
- `received_mail`: did the campaign send them mail this cycle? (1 = yes, 0 = no)
- `turned_out_2022`: did they end up voting in 2022? (1 = yes, 0 = no)

In [ ]:
voters = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/'
                     'main/weeks/wk02_observational_claims/data/voters.csv')
voters.head()

In [ ]:
voters.shape

### The naive comparison

Let's do what a vendor would do: compare the turnout rate of people who got mail to the turnout rate of people who didn't.

In [ ]:
voters.groupby('received_mail')['turned_out_2022'].mean()

Read that output: people who received mail turned out at about **0.76**, people who didn't at about **0.38**. That's a gap of roughly **37 percentage points**.

If a mail vendor showed you this comparison, she'd say: "See? Mail works. People we mailed were 37 points more likely to vote."

Before you buy that, look at one more thing.

### Compare only voters who look similar *before* the campaign ever touched them

`past_vote_score` is a pre-campaign number. It was true about each voter before anyone decided whom to mail. If we split the sample by `past_vote_score` and then look at mailed-vs-not-mailed *within* each group, we are comparing apples to apples: people whose prior voting history is the same.

The two-column `groupby` below does exactly this. It reports the 2022 turnout rate, separately, for each combination of `past_vote_score` (0–4) and `received_mail` (0 or 1).

In [ ]:
voters.groupby(['past_vote_score', 'received_mail'])['turned_out_2022'].mean()

Walk through that table row by row. Within each `past_vote_score` group, the mailed voters and the non-mailed voters turned out at **almost exactly the same rate**. Sometimes mail looks slightly better, sometimes slightly worse. The differences run in both directions, which is what noise looks like. How big does a gap have to be before you can call it noise? That is a real question and it is Week 5.

So where did the 37-point gap come from? It came from *who the campaign chose to mail*. The campaign mailed people with high past-vote scores, people who were already going to vote. The people they didn't mail were mostly people who rarely vote. The raw 37-point gap is measuring the campaign's targeting decision, not the effect of the mail itself.

The raw comparison isn't wrong. It just isn't measuring what we thought it was measuring.

This is the same mistake the vendor in this week's case is making, just more dramatic, because here we can see exactly where the gap comes from.

## Part 2: The district dataset (the vendor's chart)

Back to the case. The vendor handed you a CSV with 80 state legislative districts. Columns:

- `district_id`
- `ran_program`: did this district run her digital program in 2022?
- `turnout_2018`: what the district's turnout was the cycle *before* the program existed.
- `turnout_2022`: the outcome, i.e. what the district's turnout was in the cycle we care about.
- `median_income`, `urbanicity`: other district characteristics.

In [ ]:
districts = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/'
                        'data_science_campaigns_26/main/weeks/wk02_observational_claims/data/'
                        'district_program.csv')
districts.head()

### Reproduce the vendor's chart

Same move as Part 1: group by whether the district ran the program, and average the 2022 turnout.

In [ ]:
districts.groupby('ran_program')['turnout_2022'].mean()

Roughly **8.7 percentage points**, the exact number on the vendor's chart.

If the vendor had *flipped a coin* to decide which districts got the program, this comparison would be trustworthy. But she didn't. The campaigns picked them. So, just like in Part 1, this gap might be telling us something about the decision to run the program, not about the program itself.

### What if we subtract out what the district was doing *before* the program?

For each district, compute how much its turnout changed from 2018 to 2022. A district that was already high-turnout in 2018 starts with an advantage that has nothing to do with the vendor's program. Using the *change* instead of the 2022 level removes that baseline advantage.

In one line: make a new column called `change` equal to `turnout_2022 - turnout_2018`, then compare `change` across program vs. no-program districts.

In [ ]:
districts['change'] = districts['turnout_2022'] - districts['turnout_2018']
districts[['district_id', 'ran_program', 'turnout_2018', 'turnout_2022', 'change']].head()

In [ ]:
districts.groupby('ran_program')['change'].mean()

The 8.7-point gap has shrunk to **0.8 percentage points**. Most of what the vendor called a "lift" was just the fact that she ran her program in districts that were already turning out at higher rates in 2018.

**Does that 0.8-point remainder mean the program works, a little?**

Not really. That number is only trustworthy if we believe 2018 turnout captures *everything* about why the campaigns chose to run the program in those districts. It almost certainly doesn't. The campaigns picked those districts because they thought they were competitive, or because a donor wanted to fund them, or because the candidate was stronger, or because the field director believed in digital. None of those things are in this CSV, and we have no way to subtract them out.

**What you tell your finance director:** "Don't sign. Tell the vendor we'll pay for a pilot where *a coin flip* decides which 10 districts get the program and which 10 don't. Then we'll know." That is next week's move.

## Part 3: The same numbers, as a regression

**Nothing here is new evidence or a new fact to memorize.** It is the *same* gap you already found in Part 2, written the way papers and vendors write it. You run these cells. For now I only want you reading the **coef** column.

Researchers almost always report a comparison as a **regression** rather than a `groupby`. Let's reproduce exactly what we found, in that form, so you can read one.

### First: does 2018 turnout predict 2022 turnout?

Each dot below is one district: its 2018 turnout on the horizontal axis, its 2022 turnout on the vertical axis.

In [ ]:
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

plt.scatter(districts['turnout_2018'], districts['turnout_2022'])
plt.xlabel('2018 turnout')
plt.ylabel('2022 turnout')
plt.title('One dot = one district')
plt.show()

In [ ]:
# Correlation: one number from -1 to +1 for how tightly two columns move together.
districts['turnout_2018'].corr(districts['turnout_2022'])

That cloud of dots slopes up, and the **correlation is about 0.86**, close to its maximum of 1. A district's 2018 turnout tells you a lot about its 2022 turnout.

That's association, not causation. But it is exactly why subtracting out 2018 (Part 2) changed the gap so much. 2018 turnout is the kind of variable we wanted to adjust for.

### The vendor's gap, as a regression coefficient

Now fit a regression of 2022 turnout on whether the district ran the program. `.params` shows the two numbers the regression estimated.

In [ ]:
m1 = smf.ols('turnout_2022 ~ ran_program', data=districts).fit()
m1.params

Read those two numbers, and line them up against the `groupby` from Part 2 (next cell).

In [ ]:
# The same two numbers, two ways. Run this and compare.
print("Part 2 group averages (groupby):")
print(districts.groupby('ran_program')['turnout_2022'].mean())
print()
print("Part 3 regression coefficients (.params):")
print(m1.params)

Line them up:

- the **no-program** average (0.630) is the regression's **Intercept**;
- the **program** average (0.717) minus the no-program average is **0.087**, the regression's **ran_program** coefficient.

So the coefficient on a 0/1 variable is exactly the gap between the two group averages you already computed. The vendor's "8.7-point lift," written as a regression.

**Next week** we add a second variable to this regression and watch the coefficient move. That is where "controlling for" comes in. Today: same gap, new form.

---

## What you've seen today

- `df.shape`: rows and columns of a table.
- `df.groupby(col)[outcome].mean()`: compute an average of `outcome` separately for each value of `col`. The most important pandas move of the semester!
- `df.groupby([col1, col2])[outcome].mean()`: split by two columns at once.
- `df['new'] = df['a'] - df['b']`: create a new column from existing ones.
- `plt.scatter` + `.corr()`: see and measure how two columns move together.
- `smf.ols(...).fit()`: the same comparison as a regression. A coefficient on a 0/1 column is the group gap. (Next week: add a second variable and see what "controlling for" does.)

Next, open `wk02_problem_set.ipynb`.